# 📖 Lab 1: Upload Videos

Our first functional requirement: **Users can upload videos.**

This is deceptively simple. At YouTube's scale (~1M uploads/day, files up to 10s of GBs), we can't just `POST` a file to our API server. We need presigned URLs, direct-to-storage uploads, and a processing pipeline.

## 🏗️ Architecture — Starting Point (Naive)

```
┌────────┐  POST /upload {video + metadata}  ┌───────────────┐       ┌──────────────┐
│ Client │──────────────────────────────────>│ Upload Service │──────>│  PostgreSQL  │
│        │    10 GB video through             │ (stores file   │       │  (metadata)  │
│        │    the server! 💀                  │  on disk/S3)   │       └──────────────┘
└────────┘                                   └───────────────┘
```

Problem: The video file flows through our application server — consuming massive memory and bandwidth. At 1M uploads/day, this melts the server.

## 🏗️ Architecture — After (Presigned URLs + Processing Pipeline)

```
┌────────┐  POST /presigned_url  ┌───────────────┐  save metadata  ┌──────────────┐
│        │─────────────────────>│ Upload Service │───────────────>│  PostgreSQL  │
│ Client │                       │               │                │ (VideoMeta)  │
│        │<────────────────────│               │                └──────────────┘
└───┬────┘  {uploadUrl, videoId} └───────────────┘
    │
    │  PUT (video file via presigned URL)
    │  Multipart upload for large files
    v
┌──────────────┐   S3 event    ┌───────────────────────────────┐
│  S3 / MinIO  │──────────────>│  Video Processing Pipeline    │
│  (raw video) │               │  1. Split into segments       │
└──────────────┘               │  2. Transcode per format      │
                               │  3. Generate manifest files   │
                               │  4. Store in S3 (processed)   │
                               │  5. Update metadata (ready)   │
                               └───────────────────────────────┘
```

## Learning Objectives

- Understand why presigned URLs exist and how they work
- Upload a file directly to MinIO (S3-compatible) without going through the app server
- See the difference: naive upload vs presigned URL upload
- Build the Upload Service that generates presigned URLs and stores metadata
- Understand what video processing needs to happen (segments + formats)

## 🛠️ Setup

### 1. Start MinIO (S3-compatible) + PostgreSQL

```bash
cd system-designs/youtube
docker-compose up -d
```

### 2. Create venv & install dependencies

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python -m ipykernel install --user --name youtube --display-name "YouTube (Python)"
```

### 3. Select the **"YouTube (Python)"** kernel (top-right)

### 4. Browse (optional)
- **MinIO Console** at [http://localhost:9001](http://localhost:9001) (login: `minioadmin` / `minioadmin`)
- **Adminer** at [http://localhost:8081](http://localhost:8081) (PostgreSQL, `demo` / `demo`, database `youtube`)

In [ ]:
import psycopg2
import psycopg2.extras
from minio import Minio
import uuid
import time
import os
import io

DB_CONFIG = {
    "host": "localhost",
    "port": 5434,
    "user": "demo",
    "password": "demo",
    "database": "youtube",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

minio_client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

# Test connections
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM videos")
print(f"✅ PostgreSQL: {cur.fetchone()[0]} videos")
cur.close()
conn.close()

buckets = [b.name for b in minio_client.list_buckets()]
print(f"✅ MinIO: buckets = {buckets}")

## ❌ Naive Approach: Upload Through the Server

Let's first see the bad approach — sending the video file through our application server. This is what you'd do if you hadn't thought about presigned URLs.

In [ ]:
def upload_naive(file_data: bytes, filename: str, title: str) -> dict:
    """
    ❌ Naive: Server receives the full file, then forwards it to S3.
    The video bytes flow through our application server.
    """
    start = time.time()

    # Step 1: Server receives the entire file into memory
    file_size = len(file_data)
    print(f"  📥 Server received {file_size:,} bytes into memory")

    # Step 2: Server uploads to MinIO/S3 (proxying the bytes)
    video_id = uuid.uuid4().hex[:12]
    object_name = f"{video_id}/{filename}"

    minio_client.put_object(
        "raw-videos",
        object_name,
        io.BytesIO(file_data),
        length=file_size,
        content_type="video/mp4",
    )
    print(f"  📤 Server forwarded {file_size:,} bytes to S3")

    # Step 3: Save metadata
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO videos (id, title, user_id, status, content_type, file_size_bytes, raw_url)
        VALUES (%s, %s, %s, 'uploading', 'video/mp4', %s, %s)
    """, (video_id, title, 1, file_size, f"raw-videos/{object_name}"))
    conn.commit()
    cur.close()
    conn.close()

    duration = (time.time() - start) * 1000
    print(f"  ⏱️  Total: {duration:.0f}ms")

    return {"videoId": video_id, "approach": "naive", "ms": round(duration)}


# Create a fake 5MB "video" file
fake_video = os.urandom(5 * 1024 * 1024)  # 5 MB of random bytes
print(f"🎬 Uploading a {len(fake_video) / 1024 / 1024:.0f}MB 'video' via the naive approach:\n")

result = upload_naive(fake_video, "my_video.mp4", "My First Video")
print(f"\n📗 Result: {result}")

print(f"\n⚠️  The problem: every byte of that 5MB file flowed through our server.")
print(f"   Now imagine 10GB × 1M uploads/day = 10 PETABYTES/day through your servers.")
print(f"   Your API servers would need massive memory, bandwidth, and compute.")

## ✅ Great Approach: Presigned URLs

The fix: our server **never touches the video file**. Instead, it generates a **presigned URL** — a temporary, signed URL that allows the client to upload directly to S3/MinIO.

```
Client ──POST /presigned_url──> Server (lightweight — metadata only)
         <── {uploadUrl, videoId}

Client ──PUT video file────────> S3/MinIO (direct upload, bypasses server)
```

### What is a presigned URL?

A presigned URL is a URL with embedded authentication. It says: *"Anyone who has this URL can upload one file to this specific S3 path, but only for the next N minutes."*

- **Temporary** — expires after a set time (e.g., 1 hour)
- **Scoped** — only allows upload to a specific object key
- **Signed** — cryptographically signed by the server's S3 credentials, can't be forged

In [ ]:
from datetime import timedelta

def create_presigned_upload(title: str, description: str, content_type: str, file_size: int) -> dict:
    """
    Upload Service: POST /presigned_url

    1. Creates a VideoMetadata record in PostgreSQL (status: uploading)
    2. Generates a presigned PUT URL for the client to upload directly to S3
    3. Returns the URL + videoId — server never sees the video bytes
    """
    video_id = uuid.uuid4().hex[:12]
    object_name = f"{video_id}/original.mp4"

    # Step 1: Save metadata to PostgreSQL
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO videos (id, title, description, user_id, status, content_type, file_size_bytes, raw_url)
        VALUES (%s, %s, %s, %s, 'uploading', %s, %s, %s)
    """, (video_id, title, description, 1, content_type, file_size, f"raw-videos/{object_name}"))
    conn.commit()
    cur.close()
    conn.close()

    # Step 2: Generate presigned PUT URL (expires in 1 hour)
    upload_url = minio_client.presigned_put_object(
        "raw-videos",
        object_name,
        expires=timedelta(hours=1),
    )

    return {
        "videoId": video_id,
        "uploadUrl": upload_url,
        "objectName": object_name,
        "expiresIn": 3600,
    }


# Step 1: Server generates presigned URL (lightweight — no video data)
print("📌 Step 1: Client requests a presigned URL\n")

presigned = create_presigned_upload(
    title="My Presigned Upload",
    description="Uploaded directly to S3!",
    content_type="video/mp4",
    file_size=len(fake_video),
)

print(f"  videoId: {presigned['videoId']}")
print(f"  uploadUrl: {presigned['uploadUrl'][:80]}...")
print(f"  expiresIn: {presigned['expiresIn']}s")
print(f"\n  💡 The server only processed metadata — no video bytes touched the server!")

In [ ]:
import requests

# Step 2: Client uploads DIRECTLY to S3 using the presigned URL
print("📌 Step 2: Client uploads video directly to S3 via presigned URL\n")

start = time.time()
response = requests.put(
    presigned["uploadUrl"],
    data=fake_video,
    headers={"Content-Type": "video/mp4"},
)
upload_ms = (time.time() - start) * 1000

print(f"  HTTP {response.status_code} — {'✅ uploaded!' if response.status_code == 200 else '❌ failed'}")
print(f"  ⏱️  Upload time: {upload_ms:.0f}ms")
print(f"  📦 File size: {len(fake_video) / 1024 / 1024:.1f}MB")

# Verify it's in MinIO
stat = minio_client.stat_object("raw-videos", presigned["objectName"])
print(f"\n  🔍 Verified in MinIO: {stat.object_name} ({stat.size:,} bytes)")
print(f"\n  💡 The video went directly from client → S3.")
print(f"     Our application server was never in the data path!")

In [ ]:
# Step 3: Update metadata status to "uploaded" (in production, triggered by S3 event)
print("📌 Step 3: Update metadata status\n")

conn = get_connection()
cur = conn.cursor()
cur.execute("""
    UPDATE videos SET status = 'uploaded', updated_at = NOW()
    WHERE id = %s
""", (presigned["videoId"],))
conn.commit()

# Verify
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, title, status, file_size_bytes, raw_url FROM videos WHERE id = %s", (presigned["videoId"],))
video = cur.fetchone()
cur.close()
conn.close()

print(f"  📊 Video record:")
print(f"     id: {video['id']}")
print(f"     title: {video['title']}")
print(f"     status: {video['status']}")
print(f"     size: {video['file_size_bytes']:,} bytes")
print(f"     raw_url: {video['raw_url']}")
print(f"\n  ✅ Video metadata saved. Status: 'uploaded' → ready for processing pipeline.")

## 📊 Comparison: Naive vs Presigned URL

| | ❌ Naive (through server) | ✅ Presigned URL (direct to S3) |
|-|--------------------------|-------------------------------|
| **Data path** | Client → Server → S3 | Client → S3 (server only handles metadata) |
| **Server memory** | Must hold entire file in memory | Zero — never sees the file |
| **Server bandwidth** | Consumes upload + download bandwidth | Zero — only API metadata requests |
| **Scalability** | Limited by server resources | S3 scales independently |
| **10GB file** | Server needs 10GB RAM per concurrent upload | Server needs ~1KB per metadata request |

At 1M uploads/day with avg 500MB files, the naive approach pushes **500TB/day** through your API servers. With presigned URLs, your servers handle only lightweight metadata requests.

## 🤔 What Happens After Upload? The Video Processing Question

The raw video is now in S3. But it's **not ready to stream yet**. We need to process it. There are 3 levels of sophistication:

### ❌ Approach 1: Store the raw file only

Just serve whatever the user uploaded. Problem: different devices need different formats. An iPhone might need H.264/MP4, a browser might prefer VP9/WebM. The uploaded file may not play everywhere.

### 🟡 Approach 2: Transcode into multiple formats

Convert the raw video into multiple format versions (1080p, 720p, 480p, 360p). Better — but each format is stored as a **complete file**. The client must download the **entire** file before playing. No seeking, no adaptive bitrate.

### ✅ Approach 3: Segment + transcode + manifest

1. **Split** the video into small segments (2-10 seconds each)
2. **Transcode** each segment into multiple formats/resolutions
3. **Generate manifest files** (HLS `.m3u8` or DASH `.mpd`) that index the segments
4. Client's video player reads the manifest and streams segments on-demand

This enables:
- **Adaptive bitrate streaming** — player switches quality based on network speed
- **Seeking** — jump to any point by loading only the needed segment
- **Fast startup** — play begins as soon as the first segment loads
- **Bandwidth efficiency** — only download what you actually watch

```
Raw upload (1 file)
    │
    v
┌──────────────────────────────────────────────────────┐
│  Segment 1    Segment 2    Segment 3    ...          │
│  (0-5s)       (5-10s)      (10-15s)                  │
│                                                       │
│  Each segment transcoded to:                          │
│    1080p.mp4  720p.mp4  480p.mp4  360p.mp4           │
│                                                       │
│  Manifest files generated:                            │
│    master.m3u8 → 1080p.m3u8, 720p.m3u8, ...         │
│    Each .m3u8 lists its segment URLs                  │
└──────────────────────────────────────────────────────┘
```

> We'll build the video processing pipeline in a future lab. For now, we have the upload flow working.

## 🧹 Cleanup

In [ ]:
# Clean up test uploads (keep seed data)
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM videos WHERE user_id = 1 AND id NOT IN ('dQw4w9WgXcQ', 'jNQXAC9IVRw', '9bZkp7q19f0')")
conn.commit()
cur.close()
conn.close()

# Clean up MinIO raw-videos bucket
from minio.deleteobjects import DeleteObject
objects = minio_client.list_objects("raw-videos", recursive=True)
delete_list = [DeleteObject(obj.object_name) for obj in objects]
if delete_list:
    errors = list(minio_client.remove_objects("raw-videos", delete_list))
    print(f"✅ Cleaned {len(delete_list)} objects from raw-videos bucket.")
else:
    print("✅ raw-videos bucket already clean.")

## ✅ Summary

### What We Built

| Step | What Happens | Where |
|------|-------------|-------|
| 1. Request presigned URL | Server creates metadata + generates signed S3 URL | Upload Service → PostgreSQL |
| 2. Upload video | Client sends file directly to S3 | Client → S3/MinIO (server not involved) |
| 3. Confirm upload | S3 event triggers status update | S3 event → Upload Service → PostgreSQL |

### Key Design Decisions

| Decision | Why |
|----------|-----|
| **Presigned URLs** | Server never touches video bytes — scales to 1M uploads/day |
| **Direct-to-S3 upload** | S3/MinIO handles storage, bandwidth, durability — not our servers |
| **Metadata in PostgreSQL** | Lightweight records (~1KB each), queryable, relational |
| **Status tracking** | `uploading` → `uploaded` → `processing` → `ready` lifecycle |

### What's Next

- **Lab 2: Streaming** — how clients watch videos (manifest files, adaptive bitrate, CDN)
- **Deep dive: Video processing pipeline** — splitting, transcoding, segment storage
- **Deep dive: Resumable uploads** — multipart upload for 10GB+ files

**Pattern reference:** See `patterns/large-blobs/` for a deeper exploration of presigned URLs, resumable uploads, and blob storage patterns.